# E02 Titanic：缺失、文本特征与训练验证隔离

> Issue #9 / Epic #6 / 路线图 E02

## 目标
用 seaborn Titanic 演示真实竞赛里最常见的三类坑：
1. 缺失填充统计只能在训练集上计算
2. 姓名/舱位等文本可衍生特征，但容易泄漏
3. 提交列名常是 `Survived` 而不是 `target`

## 难点清单
- Age/Embarked/Deck 缺失机制不同
- `alone`/`alive` 等列可能造成目标泄漏（alive 几乎等于 survived）
- 训练验证隔离：用全量 median 填充会虚高线下分


In [ ]:
QUICK = True
RANDOM_STATE = 42
import warnings; warnings.filterwarnings('ignore')
from wanglaoshi import JupyterEnv as JE
JE.no_warning()


## 1. 加载数据并识别泄漏列


In [ ]:
import seaborn as sns
import pandas as pd
import numpy as np

df = sns.load_dataset('titanic').copy()
print(df.shape)
print('columns:', list(df.columns))
# alive 与 survived 高度相关，建模前应删除
leak_like = [c for c in ['alive', 'adult_male'] if c in df.columns]
print('potential leak-like columns:', leak_like)
df.head()


## 2. EDA


In [ ]:
from wanglaoshi import DataAnalyzer
eda = DataAnalyzer(df)
display(eda.missing_value_analysis())
display(eda.basic_statistics().head(15))


## 3. 正确的缺失处理：仅在训练集拟合


In [ ]:
from sklearn.model_selection import train_test_split

work = df.drop(columns=[c for c in ['alive'] if c in df.columns]).copy()
y = work['survived']
X = work.drop(columns=['survived'])

# 先只取数值列做基线（类别编码放到下一步）
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
X_num = X[num_cols]

X_train, X_valid, y_train, y_valid = train_test_split(
    X_num, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

# ✅ 正确：只用训练集统计
med = X_train.median(numeric_only=True)
X_train_f = X_train.fillna(med)
X_valid_f = X_valid.fillna(med)

# ❌ 错误示范（不要用于正式评估）：
# med_all = X_num.median(); ...
print(X_train_f.isna().sum().sum(), X_valid_f.isna().sum().sum())


## 4. 文本衍生：从 who/embark_town 等做简单编码


In [ ]:
from wanglaoshi import FeatureEngineering as FE
from sklearn.model_selection import train_test_split

# 用更完整特征：数值 + one-hot 类别（在 train 上确定列）
cat_cols = [c for c in ['sex','embarked','class','who','embark_town','deck'] if c in work.columns]
X2 = work.drop(columns=['survived'])
y2 = work['survived']
Xtr, Xva, ytr, yva = train_test_split(X2, y2, test_size=0.25, random_state=RANDOM_STATE, stratify=y2)

engineer = FE.AdvancedFeatureEngineer()
# 频率编码示例（按训练集拟合映射）
for c in cat_cols:
    freq = Xtr[c].astype(str).value_counts(normalize=True)
    Xtr[c + '_freq'] = Xtr[c].astype(str).map(freq).fillna(0)
    Xva[c + '_freq'] = Xva[c].astype(str).map(freq).fillna(0)

num2 = Xtr.select_dtypes(include=[np.number]).columns
med2 = Xtr[num2].median()
Xtr_f = Xtr[num2].fillna(med2)
Xva_f = Xva[num2].fillna(med2)
print('features', len(num2))


## 5. QuickBaseline


In [ ]:
from wanglaoshi import CompetitionTools as CT
baseline = CT.QuickBaseline()
results = baseline.run_all_models(Xtr_f, ytr, Xva_f, yva, task_type='classification')
comparison = baseline.compare_models(results, top_n=6)
comparison


## 6. 提交：目标列名为 Survived


In [ ]:
from pathlib import Path
from wanglaoshi import CompetitionTools as CT

best = comparison.iloc[0]['Model']
pred = results[best]['predictions']
out = Path('outputs/02_titanic'); out.mkdir(parents=True, exist_ok=True)
sample = pd.DataFrame({'PassengerId': np.arange(1, len(pred)+1), 'Survived': yva.to_numpy()})
sample.to_csv(out/'sample_submission.csv', index=False)
gen = CT.SubmissionGenerator()
# 不传 target_column，应自动推断 Survived
sub = gen.create_submission(pred, str(out/'sample_submission.csv'), str(out/'submission.csv'), prediction_type='label')
print(gen.validate_submission(str(out/'submission.csv'), str(out/'sample_submission.csv')))
sub.head()


## 7. 局限
- 未处理 Cabin/Deck 稀疏性与票价异常的稳健策略
- one-hot/目标编码未做折内 CV
- 下一步：用 StratifiedKFold 报告更稳的均值±方差，并显式删除泄漏列做 ablation
